# Stage3 pinned requirements GPU compatibility

Installs the bundled submission requirements into a fresh /tmp virtual environment. Enable Internet for package installation and T4 GPU. Runs the existing smoke in a new Python process; no kernel restart needed. This checks package compatibility, not the exact evaluation hardware or an offline sandbox.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, shutil, tempfile, zipfile

INPUT = Path('/kaggle/input')
WORK_PARENT = Path('/kaggle/working')
# Set SOURCE explicitly only when several matching bundles are attached.
SOURCE = None
if SOURCE is None:
    archives = list(INPUT.rglob('stage3-gpu-smoke.zip'))
    directories = [p.parent for p in INPUT.rglob('bundle.json')
                   if (p.parent/'src/stage3_pipeline.py').is_file()
                   and (p.parent/'dataset/split_manifest.csv').is_file()]
    candidates = archives + directories
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Stage3 bundle; set SOURCE to one of: {candidates}')
    SOURCE = candidates[0]
SOURCE = Path(SOURCE)
WORK = Path(tempfile.mkdtemp(prefix='stage3-gpu-', dir=WORK_PARENT))
if SOURCE.is_file():
    with zipfile.ZipFile(SOURCE) as z:
        for member in z.infolist():
            rel = PurePosixPath(member.filename)
            if rel.is_absolute() or '..' in rel.parts or '\\' in member.filename:
                raise ValueError('Unsafe ZIP member')
            if not (WORK/member.filename).resolve().is_relative_to(WORK.resolve()):
                raise ValueError('ZIP path escapes workspace')
        z.extractall(WORK)
else:
    shutil.copytree(SOURCE, WORK, dirs_exist_ok=True)
BUNDLE = json.loads((WORK/'bundle.json').read_text())
if BUNDLE.get('kind') != 'GPU_SMOKE_ONLY':
    raise ValueError('Not a Stage3 smoke bundle')
for relative, expected in BUNDLE['file_sha256'].items():
    path = (WORK/relative).resolve()
    if not path.is_relative_to(WORK.resolve()):
        raise ValueError('Manifest path escapes workspace')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Bundle checksum mismatch: {relative}')
print('Verified bundle:', SOURCE)
print('Working directory:', WORK)


In [ ]:
import subprocess, sys, time, os
VENV = Path(tempfile.mkdtemp(prefix='stage3-pinned-', dir='/tmp'))
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
started = time.monotonic()
with (WORK/'install.log').open('w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-cache-dir', '-r', str(WORK/'requirements.txt')], stdout=log, stderr=subprocess.STDOUT)
INSTALL = {'exit_code': installed.returncode, 'seconds': time.monotonic()-started,
           'requirements_sha256': hashlib.sha256((WORK/'requirements.txt').read_bytes()).hexdigest()}
(WORK/'install.json').write_text(json.dumps(INSTALL, indent=2))
if installed.returncode:
    print((WORK/'install.log').read_text()[-12000:])
    raise RuntimeError('Pinned requirements installation failed')
print('Installation:', INSTALL)
RUNNER = 'from pathlib import Path\nimport sys, json, hashlib, zipfile\nWORK=Path(sys.argv[1])\nBUNDLE=json.loads((WORK/"bundle.json").read_text())\nimport importlib.metadata\nversions={}\nfor line in (WORK/"requirements.txt").read_text().splitlines():\n    if not line.strip() or line.startswith("#"): continue\n    name,expected=line.strip().split("==")\n    actual=importlib.metadata.version(name)\n    assert actual.split("+")[0]==expected, (name,expected,actual)\n    versions[name]=actual\n(WORK/"pinned-versions.json").write_text(json.dumps(versions,indent=2))\nimport sys, platform, importlib.metadata\nimport torch, torchvision, numpy, pandas, cv2\ndef FileLink(path): return path\ndef display(value): print(value)\n\nENVIRONMENT = {\n    \'python\': sys.version,\n    \'platform\': platform.platform(),\n    \'torch\': str(torch.__version__),\n    \'torchvision\': str(torchvision.__version__),\n    \'numpy\': numpy.__version__, \'pandas\': pandas.__version__,\n    \'opencv\': cv2.__version__,\n    \'cuda_available\': torch.cuda.is_available(),\n    \'torch_cuda\': torch.version.cuda,\n    \'gpu_names\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n}\nprint(json.dumps(ENVIRONMENT, indent=2))\nif not ENVIRONMENT[\'cuda_available\']:\n    raise RuntimeError(\'Enable a Kaggle GPU and use a CUDA-enabled PyTorch installation.\')\n(WORK/\'environment.json\').write_text(json.dumps(ENVIRONMENT, indent=2))\nprint(\'Baseline requirements:\')\nprint((WORK/\'requirements.txt\').read_text())\n\nimport subprocess, time\n\ndef run_logged(arguments, name):\n    started = time.monotonic()\n    with (WORK/name).open(\'w\', encoding=\'utf-8\') as log:\n        process = subprocess.Popen([sys.executable, *arguments], cwd=WORK,\n                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                   text=True, bufsize=1)\n        for line in process.stdout:\n            print(line, end=\'\')\n            log.write(line)\n        returncode = process.wait()\n    if returncode:\n        raise RuntimeError(f\'{name} failed with exit code {returncode}; retain {WORK/name}\')\n    return {\'exit_code\': returncode, \'seconds\': time.monotonic()-started}\n\nRUNS = {}\nRUNS[\'tests\'] = run_logged([\'src/test_stage3_pipeline.py\'], \'tests.log\')\nRUNS[\'audit\'] = run_logged([\'src/stage3_pipeline.py\', \'audit\', \'--dataset-dir\', \'dataset\'], \'audit.log\')\nRUNS[\'smoke\'] = run_logged([\'src/stage3_pipeline.py\', \'smoke\', \'--dataset-dir\', \'dataset\',\n                          \'--output-dir\', \'smoke-run\', \'--device\', \'cuda\'], \'smoke.log\')\n(WORK/\'execution.json\').write_text(json.dumps(RUNS, indent=2))\n\nreport = json.loads((WORK/\'smoke-run/smoke_report.json\').read_text())\npredictions = pandas.read_csv(WORK/\'smoke-run/predictions.csv\')\nif report.get(\'status\') != \'PASS\' or report.get(\'device\') != \'cuda\' or report.get(\'public_cuda_entrypoint_tested\') is not True:\n    raise RuntimeError(\'GPU/public entrypoint verification did not pass\')\nif list(predictions.columns) != [\'ID\',\'sample_index\',\'accel_label\',\'steer_label\']:\n    raise RuntimeError(\'Prediction schema mismatch\')\nif predictions[\'ID\'].tolist() != [\'SMOKE_S3\']*4 or predictions.sample_index.tolist() != list(range(4)):\n    raise RuntimeError(\'Missing or extra smoke frames\')\nRESULT = WORK/\'stage3-gpu-result.zip\'\nfiles = [\'environment.json\',\'execution.json\',\'bundle.json\',\'tests.log\',\'audit.log\',\'smoke.log\',\n         \'smoke-run/smoke_report.json\',\'smoke-run/predictions.csv\']\nwith zipfile.ZipFile(RESULT, \'x\', compression=zipfile.ZIP_DEFLATED) as z:\n    for relative in files:\n        z.write(WORK/relative, relative)\nprint(\'GPU SMOKE PASS — not full training\')\nprint(json.dumps(report, indent=2))\ndisplay(FileLink(str(RESULT)))\n\n'
(WORK/'compatibility_runner.py').write_text(RUNNER)
with (WORK/'compatibility.log').open('w') as log:
    completed = subprocess.run([PYTHON, str(WORK/'compatibility_runner.py'), str(WORK)], cwd=WORK, stdout=log, stderr=subprocess.STDOUT)
print((WORK/'compatibility.log').read_text()[-12000:])
if completed.returncode:
    raise RuntimeError('Pinned GPU smoke failed; inspect compatibility.log')
with zipfile.ZipFile(WORK/'stage3-compatibility-evidence.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in ['install.json','install.log','pinned-versions.json','compatibility_runner.py','compatibility.log','requirements.txt']:
        archive.write(WORK/name,name)
print('PASS:', WORK/'stage3-gpu-result.zip')
